# 4. Segmentazione dei possessori di BEV

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedeGambe/Master_s_thesis_Data_science/blob/main/notebooks/04_cluster.ipynb)

Come nella tesi, la segmentazione finale usa il K-Mode sulle variabili categoriali (k = 4). Il K-Means sugli indici compositi e il clustering gerarchico servono da verifica.

In [ ]:
# Configurazione dell'ambiente (locale o Google Colab)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Master_s_thesis_Data_science"):
        !git clone -q -b main https://github.com/FedeGambe/Master_s_thesis_Data_science.git
    %cd Master_s_thesis_Data_science
    !pip install -q -r requirements-colab.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import plotly.io as pio
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)
if os.environ.get("TESI_GRAFICI_STATICI"):
    pio.renderers.default = "png"  # usato solo per salvare output visibili su GitHub
    pio.defaults.default_width = 1000

In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tesi_bev import config as C, dati, esplorativa as E, cluster as K, grafici as G

bev = dati.solo_bev(dati.crea_variabili_categoriali())
df = K.dataset_cluster(bev, E.indici_compositi(bev))
K.VARIABILI_KMODE

## 4.1 K-Mode: scelta del numero di cluster

Il K-Mode usa la distanza di Hamming (numero di caratteristiche diverse) e rappresenta ogni cluster con la modalità
più frequente di ciascuna variabile. Il costo cala in modo regolare: k = 4 è scelto, come nella tesi, per avere
profili ancora interpretabili.

In [ ]:
costo = K.costo_kmode(df[K.VARIABILI_KMODE], range(1, 9))
G.gomito_silhouette(costo.rename(columns={"costo": "wss"}).assign(silhouette=None)).show()
costo

## 4.2 K-Mode con k = 4: i profili (circa 1 minuto)

In [ ]:
km_mode = K.kmode(df, k=4)
df["cluster_kmode"] = km_mode["etichette"]
print(f"Costo {km_mode['costo']:.0f} - dimensioni {df['cluster_kmode'].value_counts().sort_index().tolist()}")
km_mode["centri"].T

In [ ]:
profilo_num = ["Età", C.REDDITO, C.N_PERSONE, C.N_AUTO, C.VMT, C.N_VIAGGI_LUNGHI, C.DISTANZA_LAVORO, C.EMISSIONI]
df.groupby("cluster_kmode")[profilo_num].mean().round(1)

## 4.3 Stabilità del K-Mode

Molte partizioni hanno un costo quasi uguale: ripetendo l'algoritmo con semi diversi i gruppi cambiano. Un ARI vicino
a 0 indica partizioni molto diverse, quindi i profili vanno letti come **segmenti descrittivi**, non come gruppi
naturali. Il calcolo richiede alcuni minuti.

In [ ]:
CALCOLA_STABILITA = False
if CALCOLA_STABILITA:
    print(f"ARI medio tra semi diversi: {K.stabilita_kmode(df, k=4):.3f}")

## 4.4 Verifica con il K-Means (configurazione della tesi)

In [ ]:
X = df[K.VARIABILI_KMEANS]
scelta = K.gomito_silhouette(X, range(2, 9))
G.gomito_silhouette(scelta).show()
km = K.kmeans(X, k=4)
print(f"WSS = {km['wss']:.0f} (tesi, in R: 34.630) - silhouette = {km['silhouette']:.3f} (tesi: 0,395)")
coord = PCA(2).fit_transform(StandardScaler().fit_transform(X))
G.cluster_2d(coord, km["etichette"]).show()

Le 5 variabili dummy dell'auto precedente, una volta standardizzate, dominano la distanza euclidea: i 4 cluster
coincidono quasi esattamente con il tipo di auto posseduta in precedenza. È il motivo per cui nella tesi il K-Means è
stato scartato a favore del K-Mode.

In [ ]:
print(f"ARI tra K-Means e auto precedente: {adjusted_rand_score(df[C.AUTO_PRECEDENTE], km['etichette']):.3f}")
print(f"ARI tra K-Means e K-Mode: {adjusted_rand_score(df['cluster_kmode'], km['etichette']):.3f}")
G.composizione_cluster(df, km["etichette"], C.AUTO_PRECEDENTE).show()
pd.crosstab(km["etichette"], df[C.AUTO_PRECEDENTE])

## 4.5 K-Means senza le dummy dell'auto precedente

In [ ]:
variabili = [v for v in K.VARIABILI_KMEANS if not v.startswith("Auto precedente")] + ["Status socioeconomico", C.EMISSIONI]
X2 = df[variabili]
G.gomito_silhouette(K.gomito_silhouette(X2, range(2, 9))).show()
km2 = K.kmeans(X2, k=4)
print(f"Silhouette: {km2['silhouette']:.3f} - stabilità (ARI bootstrap): {K.stabilita_bootstrap(X2, 4, 20):.3f}")
K.profilo_cluster(df, km2["etichette"], ["Età", C.REDDITO, C.EMISSIONI, C.N_AUTO, C.VMT, C.N_VIAGGI_LUNGHI],
                  [C.CLASSE_REDDITO, C.ISTRUZIONE, C.CASA_INDIPENDENTE, C.AUTO_PRECEDENTE])

Anche senza le dummy non emerge una struttura a gruppi netta, coerentemente con l'instabilità del K-Mode.

## 4.6 Clustering gerarchico (Ward)

In [ ]:
h = K.gerarchico(X, k=4)
print(f"Gerarchico Ward: correlazione cofenetica {h['correlazione_cofenetica']:.3f}, silhouette {h['silhouette']:.3f}, "
      f"ARI con K-Means {adjusted_rand_score(km['etichette'], h['etichette']):.3f}")